# Reproducability and Seeding
This notebook showcases how TorchSig handles random seeding to allow reproducable experiments.

---

## Iterable Dataset
The TorchSigIterableDataset class inherits torch IterableDataset, and is used to sample synthetic datasets at runtime

In [1]:
from torchsig.datasets.datasets import TorchSigIterableDataset
from torchsig.utils.defaults import TorchSigDefaults

dataset_metadata = TorchSigDefaults().default_dataset_metadata
print(dataset_metadata)

{'num_iq_samples_dataset': 262144, 'num_signals_min': 1, 'num_signals_max': 1, 'fft_size': 512, 'fft_stride': 512, 'sample_rate': 10000000, 'noise_power_db': 0.0, 'snr_db_min': 0.0, 'snr_db_max': 50.0, 'cochannel_overlap_probability': 0.2, 'signal_duration_in_samples_min': 209715.2, 'signal_duration_in_samples_max': 262144.0, 'bandwidth_min': 2500000, 'bandwidth_max': 3333333, 'signal_center_freq_min': -2500000, 'signal_center_freq_max': 2499999, 'frequency_min': -2500000, 'frequency_max': 2499999}


Creating the dataset without seeding; this print statement will create a different random signal every time you call it.

If you run this cell multiple times, or if you reload this notebook and run it again it will not produce the same signal.

In [2]:
dataset = TorchSigIterableDataset(metadata=dataset_metadata)
print(next(dataset).data)
print(next(dataset).data)
print(next(dataset).data)

[ 0.03417929+0.05571558j  0.13798922-0.03371051j -0.04852524-0.03049647j
 ... -0.01916722+0.05816604j -0.09601081+0.03380658j
 -0.00775587+0.05150901j]


[-0.13448474-0.0550117j  -0.00816097-0.00702455j  0.02602751-0.08339459j
 ... -0.02209694-0.01047525j  0.06176484+0.01381563j
 -0.1368908 +0.09102734j]


[ 0.11791452+0.02568829j -0.02838005+0.06324411j -0.00081617-0.00238937j
 ...  0.01613826-0.01788137j -0.00337055-0.00512317j
 -0.08663452-0.01879739j]


In [3]:
dataset = TorchSigIterableDataset(metadata=dataset_metadata)
print(next(dataset).data)
print(next(dataset).data)
print(next(dataset).data)

[ 0.00465816+0.09598327j  0.07469516-0.08112895j -0.0356834 -0.11606906j
 ... -0.0602401 -0.02098601j -0.01423639-0.00831797j
  0.07264906-0.00094667j]


[-0.0345399 -0.12708963j  0.00267879+0.07197642j -0.00299409-0.01840857j
 ... -0.07584412-0.0092598j  -0.01145767-0.0658598j
  0.02212182+0.05562194j]
[-0.00568168+0.00114346j -0.08009298-0.04564496j  0.11139882-0.17064747j
 ...  0.07633691-0.11218905j  0.05695221+0.00591833j
  0.01773868+0.03198662j]


## Seeding
All torchsig Transforms, Datasets, DatasetMetadata objects, and DataLoaders are seedable objects.

This means they all have a .seed(N) method, which can be called to input a random seed. If no seed is given, the seedable object with produce its own seed and generate different random numbers every time you run your code.

If you want reproducable experiments, you generally will want to call .seed(N) on some integer N of your choosing. This will ensure the same 'random' outcomes occur each time the code is executed.

You don't need to seperately seed connected objects. If a dataset contains several transforms, seeding the dataset is enough to also correctly seed all of its transforms.

In general, you will only need to call .seed() on the top level object you are using (typically either a dataset or a data loader).

NOTE: Calling numpy.random.seed will not seed torchsig datasets; they should always be seeded explicitely if a seed is desired

### Seeding the Dataset
Here the same dataset from above is seeded; this code will produce the same random signals every time it is run

In [4]:
dataset = TorchSigIterableDataset(metadata=dataset_metadata)
dataset.seed(42)
print(next(dataset).data)
print(next(dataset).data)
print(next(dataset).data)

[ 0.02311219-0.02543635j -0.07888075+0.02726327j  0.05692025-0.10656164j
 ...  0.13700777+0.03362937j  0.00722697+0.03663129j
  0.02593248+0.05931827j]


[-0.10525516+0.08490815j -0.07363532-0.10551274j  0.1273497 -0.0326958j
 ...  0.00961168-0.03739209j  0.02748841+0.05255114j
 -0.03916043+0.0092365j ]


[ 0.09063724-0.01121376j -0.06297278+0.1473536j  -0.02430215-0.02785327j
 ... -0.12130772-0.12061954j -0.10326934+0.02931302j
  0.10345804-0.11350849j]


In [5]:
dataset = TorchSigIterableDataset(metadata=dataset_metadata)
dataset.seed(42)
print(next(dataset).data)
print(next(dataset).data)
print(next(dataset).data)

[ 0.02311219-0.02543635j -0.07888075+0.02726327j  0.05692025-0.10656164j
 ...  0.13700777+0.03362937j  0.00722697+0.03663129j
  0.02593248+0.05931827j]
[-0.10525516+0.08490815j -0.07363532-0.10551274j  0.1273497 -0.0326958j
 ...  0.00961168-0.03739209j  0.02748841+0.05255114j
 -0.03916043+0.0092365j ]


[ 0.09063724-0.01121376j -0.06297278+0.1473536j  -0.02430215-0.02785327j
 ... -0.12130772-0.12061954j -0.10326934+0.02931302j
  0.10345804-0.11350849j]


### Seeding a DataLoader
On a single worker threads/process, seeding a dataset alone is sufficient to reproduce results correctly.

Since DataLoaders typically use several different worker threads/processes, each with a copy of the dataset, we generally want each worker to have a different seed for its copy of the dataset, so that its randomly generated data does not match that of the other workers.

This is unneccessary for loading static data from files, but it is needed for loading on-the-fly random generated data correctly.

To address this issue, TorchSig exposes a WorkerSeedingDataLoader, which will seed a torchsig dataset differently in all workers.

NOTE: WorkerSeedingDataLoader uses it's own worker init function, and is not compatible with other custom worker init functions; the exact data generated will still depend on the configuration of workers, so it will not produce the same data with different worker counts


In the code below, we create and seed a WorkerSeedingDataLoader for our dataset

In [6]:
from torchsig.utils.data_loading import WorkerSeedingDataLoader

batch_size = 8
num_workers = 2

In [7]:
dataset = TorchSigIterableDataset(metadata=dataset_metadata)
dataloader = WorkerSeedingDataLoader(
    dataset, batch_size=batch_size, collate_fn=lambda x: x
)
dataloader.seed(42)

data = [x.data for x in next(iter(dataloader))]
print(data)

[array([ 0.02311219-0.02543635j, -0.07888075+0.02726327j,
        0.05692025-0.10656164j, ...,  0.13700777+0.03362937j,
        0.00722697+0.03663129j,  0.02593248+0.05931827j],
      shape=(262144,), dtype=complex64), array([-0.10525516+0.08490815j, -0.07363532-0.10551274j,
        0.1273497 -0.0326958j , ...,  0.00961168-0.03739209j,
        0.02748841+0.05255114j, -0.03916043+0.0092365j ],
      shape=(262144,), dtype=complex64), array([ 0.09063724-0.01121376j, -0.06297278+0.1473536j ,
       -0.02430215-0.02785327j, ..., -0.12130772-0.12061954j,
       -0.10326934+0.02931302j,  0.10345804-0.11350849j],
      shape=(262144,), dtype=complex64), array([ 0.0539151 +0.02831647j,  0.06428367-0.0236385j ,
        0.09936596-0.01942791j, ...,  0.00289481-0.03654752j,
        0.04346073-0.10604122j, -0.02921288+0.11209081j],
      shape=(262144,), dtype=complex64), array([-0.17006281+0.12345932j,  0.01307517-0.02773295j,
        0.06182327-0.04399231j, ..., -0.14248742+0.018505j  ,
       -

Because we are seeding our dataloader with the same seed value, and because both dataloaders have the same worker count, this code will produce the same batch of signals every time it is run

In [8]:
dataset = TorchSigIterableDataset(metadata=dataset_metadata)
dataloader = WorkerSeedingDataLoader(
    dataset, batch_size=batch_size, collate_fn=lambda x: x
)
dataloader.seed(42)

data = [x.data for x in next(iter(dataloader))]
print(data)

[array([ 0.02311219-0.02543635j, -0.07888075+0.02726327j,
        0.05692025-0.10656164j, ...,  0.13700777+0.03362937j,
        0.00722697+0.03663129j,  0.02593248+0.05931827j],
      shape=(262144,), dtype=complex64), array([-0.10525516+0.08490815j, -0.07363532-0.10551274j,
        0.1273497 -0.0326958j , ...,  0.00961168-0.03739209j,
        0.02748841+0.05255114j, -0.03916043+0.0092365j ],
      shape=(262144,), dtype=complex64), array([ 0.09063724-0.01121376j, -0.06297278+0.1473536j ,
       -0.02430215-0.02785327j, ..., -0.12130772-0.12061954j,
       -0.10326934+0.02931302j,  0.10345804-0.11350849j],
      shape=(262144,), dtype=complex64), array([ 0.0539151 +0.02831647j,  0.06428367-0.0236385j ,
        0.09936596-0.01942791j, ...,  0.00289481-0.03654752j,
        0.04346073-0.10604122j, -0.02921288+0.11209081j],
      shape=(262144,), dtype=complex64), array([-0.17006281+0.12345932j,  0.01307517-0.02773295j,
        0.06182327-0.04399231j, ..., -0.14248742+0.018505j  ,
       -

Tensors expect each array to have the same size. This presents a challenge when there is no signal present, ex: `num_signals_min == 0`, which is the default for `DatasetMetadata()`. All of the IQ time-series within a dataset will always be same, even when there is no signal present, because the underlying noise is created. However, when there is no signal the metadata becomes an empty list. Therefore when viewing the metadata in tensor format there will be zero-padded fields. When processing the output of the data loader the zero-padding will need to be undone.